# 15 · Voronoi catchments, checked against shapely

## What this checks

**Voronoi catchments** divides the ground into cells, one around each site. Each cell is the ground nearer its site
than any other, in a straight line. This check covers release 0.5.0.

This is the first step of the John Snow example. It was the one step of that example with no check. With it, the
whole example is now checked independently: the cells, the clip and the count.

## How it is checked

Two unrelated methods draw the same cells:

- **Fieldwork** starts from a rectangle. For each other site, it cuts away the ground nearer that site. Distances are
  in metres, on a flat projection.
- **The reference** asks **shapely**, the Python library for shapes, for the Voronoi diagram of the same points.
  Shapely uses GEOS, the geometry library behind most GIS software. The diagram is then cut to the same rectangle.

Neither method informs the other, and no code is shared with Fieldwork. This is the strongest kind of comparison
this lab makes.

The rectangle is worked out again here from the reporting area, by the rule Fieldwork states. That rule takes the
corners of the area's bounding box in metres, their furthest east, west, north and south, and adds 100 m. So that
step is tested too.

The data are 8 pumps from the John Snow example.

## What should happen

1. **Each cell** matches GEOS's cell for the same site within 1 square metre. The difference is measured as the area
   in one cell and not the other.
2. **The cells fill the rectangle.** Together they cover it within 1 square metre, and no two overlap.
3. **Each cell holds its own site.** That is what a Voronoi cell is for.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_15_voronoi import run

report = run(Path.cwd() / "fixtures" / "voronoi-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["sites"], "sites")
print(f"worst symmetric difference   {report['worstSymmetricDifferenceSquareMetres']:.3e} m2")
print(f"union against the envelope   {report['unionAgainstEnvelopeSquareMetres']:.3e} m2")
print(f"pairwise overlap             {report['pairwiseOverlapSquareMetres']:.3e} m2")
print(f"every cell holds its site    {report['everyCellHoldsItsSite']}")
print()
print(f"{'site':<12} {'cell area m2':>14} {'symmetric difference':>22}")
for row in report["perCell"]:
    print(f"{row['siteId']:<12} {row['areaSquareMetres']:14.0f} {row['symmetricDifferenceSquareMetres']:22.3e}")

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.5.0:
**Agrees**.

- **Each cell.** All 8 cells were within about 2.5 × 10⁻⁶ square metres of GEOS's cells.
- **Filling the rectangle.** Together they missed the rectangle by about 2.1 × 10⁻⁶ square metres. They overlapped
  by about 6.0 × 10⁻⁷ square metres.
- **Own sites.** Every cell held its own site.

These are a few millionths of a square metre, far inside the 1 square metre allowed.

Running the cell above repeats the check on the same data.

## What this does not show

- That a nearest-pump cell shows which pump anybody drank from. Straight-line distance on a map ignores streets,
  walls and the Thames. The reference ignores them in the same way.
- That the cells mean anything because they agree. Two methods dividing the ground the same way is agreement about
  geometry, not about where people went.